# 01 — Introduction & Setup

Welcome to a practical course on **LangChain**, built around Google's **Gemini** models.

This course assumes you already know Python and object-oriented programming, so we
won't spend time on those. Instead we focus on the one new skill: assembling language
models into useful, reliable applications.

In this first notebook you will:

1. Understand what LangChain actually is (and what it is *not*).
2. See how the LangChain packages fit together.
3. Get a Gemini API key working from a notebook.
4. Make your first model call and read the response.

## 1. What problem does LangChain solve?

Calling a single LLM is easy. The provider gives you an SDK, you send text, you get text back:

```python
# pseudo-code for a raw provider SDK
client.generate("Summarize this email: ...")
```

Real applications are rarely a single call. They look more like:

- *Take a user's question, look up relevant documents, feed them to the model, and
  cite the sources.*
- *Ask the model to return a strict JSON object so the rest of the program can use it.*
- *Let the model call your functions (search, calculator, database) and loop until the
  task is done.*
- *Remember the conversation so far.*

**LangChain** is a framework for building those applications. It gives you:

- **A standard interface** over many model providers (Gemini, OpenAI, Anthropic, local
  models...). Swap the model line and the rest of your code is unchanged.
- **Composable building blocks** — prompts, models, parsers, retrievers, tools — that
  snap together with a single `|` operator.
- **Higher-level constructs** — retrieval pipelines (RAG), memory, and agents — so you
  don't reinvent them.

It is *not* a model itself and not a hosting service. Think of it as the wiring and the
standard parts catalog for LLM apps.

## 2. The package ecosystem

LangChain is split into focused packages. You install only what you need.

| Package | Role |
|---------|------|
| `langchain-core` | The base abstractions: messages, prompts, output parsers, the `Runnable` interface that makes `\|` work. Almost everything depends on it. |
| `langchain` | Higher-level orchestration: `init_chat_model`, agents (`create_agent`), common chains. |
| `langchain-google-genai` | The **integration** package for Gemini. Provides `ChatGoogleGenerativeAI` and `GoogleGenerativeAIEmbeddings`. |
| `langchain-community` | Community-maintained integrations: many document loaders, vector stores, tools. |
| `langchain-text-splitters` | Utilities to chop documents into chunks (used in retrieval). |
| `langgraph` | The engine for stateful, multi-step agents. LangChain 1.0 agents are built on it. |

A useful rule of thumb: **provider-specific code lives in its own package**
(`langchain-google-genai`), while the **generic concepts** live in `langchain-core`.
That separation is exactly what lets you switch providers later.

## 3. Why Gemini, and which model?

Google's **Gemini** family is a strong, widely available set of models with a generous
free tier — ideal for learning. The two you'll use most:

- **`gemini-2.5-flash`** — fast and cheap. This is our default for almost everything in
  the course.
- **`gemini-2.5-pro`** — slower and pricier, but stronger at hard reasoning. Reach for it
  when Flash struggles.

Model names evolve over time (newer generations appear regularly). If a name ever stops
working, check the current list in
[Google AI Studio](https://aistudio.google.com/) and swap the string — nothing else in
your code changes. That swappability is the whole point of the standard interface.

## 4. Get your API key

1. Go to [Google AI Studio → API keys](https://aistudio.google.com/app/apikey).
2. Create a key (the free tier is enough for this course).
3. In the course folder, copy `.env.example` to a file named `.env` and paste your key:

   ```
   GOOGLE_API_KEY=AIza...your-real-key...
   ```

The `.env` file is already in `.gitignore`, so you won't accidentally commit your key.

The cell below loads that file. `python-dotenv`'s `load_dotenv()` reads `.env` from the
current folder and puts its values into `os.environ`, which is exactly where
LangChain's Gemini classes look for the key.

In [7]:
import os
from dotenv import load_dotenv

# Reads the .env file in this folder and loads GOOGLE_API_KEY into os.environ.
load_dotenv()

assert os.environ.get("GOOGLE_API_KEY"), (
    "GOOGLE_API_KEY not found.\n"
    "Copy .env.example to .env and paste your Gemini key, then re-run this cell."
)
print("API key loaded. You're ready to call Gemini.")

API key loaded. You're ready to call Gemini.


## 5. Your first model call

`ChatGoogleGenerativeAI` is the class that talks to Gemini. You create one instance
(it's reusable) and call `.invoke()` with your input.

A few constructor arguments worth knowing now:

- `model` — which Gemini model to use.
- `temperature` — randomness. `0` is focused and repeatable; higher (up to ~1) is more
  creative. We'll explore this properly in notebook 02.

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
)

response = model.invoke("In two sentences, explain what LangChain is to a Python developer.")
print(response.content)

LangChain is a Python framework designed to simplify the development of applications powered by large language models (LLMs). It provides modular abstractions for interacting with LLMs, managing prompts, integrating external data sources, and orchestrating complex chains or agents to build sophisticated AI applications.


### What did we get back?

`.invoke()` does not return a plain string. It returns an **`AIMessage`** object. The
text is in `.content`, but the object carries more — which is why LangChain uses a
message type instead of a bare string.

Run the next cell to inspect it.

In [11]:
print(type(response))
print("\n--- content ---")
print(response.content)
print("\n--- response_metadata (model, finish reason, safety) ---")
print(response.response_metadata)
print("\n--- usage_metadata (token counts) ---")
print(response.usage_metadata)

<class 'langchain_core.messages.ai.AIMessage'>

--- content ---
LangChain is a Python framework designed to simplify the development of applications powered by large language models (LLMs). It provides modular abstractions for interacting with LLMs, managing prompts, integrating external data sources, and orchestrating complex chains or agents to build sophisticated AI applications.

--- response_metadata (model, finish reason, safety) ---
{'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}

--- usage_metadata (token counts) ---
{'input_tokens': 15, 'output_tokens': 724, 'total_tokens': 739, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 671}}


The `usage_metadata` (input/output token counts) is how you track cost. We'll come
back to messages and metadata in depth in notebook 02.

## 6. An alternative: `init_chat_model`

Instead of importing a provider class directly, `langchain` offers `init_chat_model`,
a factory that builds the right model object from a string. This is handy when you want
the provider to be configurable rather than hard-coded.

Both styles are equivalent; use whichever reads better for your project. We'll mostly
use the explicit `ChatGoogleGenerativeAI` import so it's always clear which provider
is in play.

In [12]:
from langchain.chat_models import init_chat_model

# "google_genai:gemini-2.5-flash" = provider:model
model2 = init_chat_model("google_genai:gemini-2.5-flash", temperature=0)

print(model2.invoke("Name three things LangChain can help build.").content)

LangChain is a framework designed to simplify the creation of applications powered by large language models (LLMs). Here are three things it can help build:

1.  **Conversational AI Applications / Intelligent Chatbots:** LangChain provides tools for managing conversation history (memory), chaining together multiple prompts, and integrating with various LLMs, making it ideal for building sophisticated chatbots that can maintain context over multiple turns.
2.  **Question-Answering (QA) Systems over Custom Data (RAG):** LangChain excels at Retrieval-Augmented Generation (RAG). It helps you load, process, embed, and retrieve information from your own documents, databases, or APIs, allowing an LLM to answer questions based on your specific, private, or proprietary data.
3.  **Autonomous Agents that Perform Actions:** LangChain's agent capabilities allow LLMs to choose and use external tools (like search engines, calculators, APIs, or even custom Python functions) to achieve a goal. This en

## 7. A slightly more useful example

Even with just a model, you can do real work. Here we ask Gemini to act as a code
reviewer. Notice we're still only using `.invoke()` — but the *instruction* does the
heavy lifting. In the next notebooks we'll make prompts like this reusable and reliable.

In [13]:
review_prompt = """You are a senior Python reviewer. Review the function below.
List up to three concrete improvements as a numbered list. Be brief.

def calc(l):
    s = 0
    for i in range(len(l)):
        s = s + l[i]
    return s / len(l)
"""

print(model.invoke(review_prompt).content)

Here are three concrete improvements:

1.  **Use `sum()` built-in:** Replace the manual summation loop with `s = sum(l)` for conciseness and readability.
2.  **Handle empty list:** Add a check for an empty list (`if not l:`) to prevent a `ZeroDivisionError`.
3.  **Descriptive naming:** Use more descriptive names for the function (e.g., `calculate_average`) and its parameter (e.g., `numbers` or `data`).


## Your turn

Five exercises drawn from everyday tasks you might hand to a model — a commit-message
writer, a cost meter, a support triager, and more. Use the blank cells below to attempt each
one before opening its solution.

**Exercise 1 — Commit-message writer.** Write a helper `commit_message(change: str) -> str`
that turns a plain description of code changes into a single-line
[Conventional Commit](https://www.conventionalcommits.org) (e.g. `feat: add password reset`).
Try it on `"added a password-reset email and fixed a typo on the login form"`.

*Sample run (illustrative — your wording will vary):*

```text
feat: add password-reset email and fix login form typo
```

<details><summary>Show solution</summary>

```python
def commit_message(change: str) -> str:
    prompt = (
        "Turn this description of code changes into ONE Conventional Commit "
        "line (type: summary). Return only that line.\n\n" + change
    )
    return model.invoke(prompt).content.strip()

print(commit_message("added a password-reset email and fixed a typo on the login form"))
```

</details>

**Exercise 2 — Focused vs. creative.** Build two models, `precise` (`temperature=0`) and
`creative` (`temperature=0.9`). Ask both for five product-name ideas for a budgeting app,
twice each. Which one keeps repeating itself, and why?

*Sample run (illustrative):*

```text
precise  1: BudgetBuddy, PennyWise, CashTrack, SaveSmart, FundFlow
precise  2: BudgetBuddy, PennyWise, CashTrack, SaveSmart, FundFlow
creative 1: PennyPilot, Cashly, BudgetBee, MintMate, WalletWise
creative 2: CoinKeeper, SpendLens, ThriftIQ, BudgetNest, FinFox
```

<details><summary>Show solution</summary>

```python
precise  = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
creative = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9)
q = "Five product-name ideas for a budgeting app. Comma-separated, names only."
print("precise  1:", precise.invoke(q).content)
print("precise  2:", precise.invoke(q).content)
print("creative 1:", creative.invoke(q).content)
print("creative 2:", creative.invoke(q).content)
```

`temperature=0` is near-deterministic, so `precise` tends to repeat; `creative` varies.

</details>

**Exercise 3 — Token & cost meter.** Write `report(prompt: str)` that calls the model,
prints the input/output token counts from `usage_metadata`, and estimates the cost given an
example output price of \$0.0003 per 1K tokens. Compare a short prompt with a long one.

*Sample run (illustrative — token counts vary):*

```text
in=   4  out=   9  ~$0.00000
in=  16  out= 198  ~$0.00006
```

<details><summary>Show solution</summary>

```python
def report(prompt: str, out_price_per_1k=0.0003) -> str:
    resp = model.invoke(prompt)
    u = resp.usage_metadata
    cost = u["output_tokens"] / 1000 * out_price_per_1k
    print(f"in={u['input_tokens']:>4}  out={u['output_tokens']:>4}  ~${cost:.5f}")
    return resp.content

report("Say hello.")
report("Explain how HTTPS keeps a web request private, in two paragraphs.")
```

</details>

**Exercise 4 — One-line model swap.** Use `init_chat_model` with a single `MODEL` string to
answer a reasoning puzzle. Show that upgrading Flash → Pro means changing only that one
string — nothing else in the code.

*Sample run:*

```text
0.05
```

<details><summary>Show solution</summary>

```python
from langchain.chat_models import init_chat_model

MODEL = "google_genai:gemini-2.5-flash"   # later: "google_genai:gemini-2.5-pro"
llm = init_chat_model(MODEL, temperature=0)
print(llm.invoke(
    "A bat and ball cost $1.10. The bat costs $1.00 more than the ball. "
    "How much is the ball? Answer with the number only."
).content)
```

Only the `MODEL` string changes to swap models — the rest of the app is untouched.

</details>

**Exercise 5 — Support-message triage.** Write `triage(message: str) -> str` that labels a
customer message as `billing`, `technical`, or `general` **and** drafts a one-sentence
reply. Try it on `"I was charged twice for my subscription this month."`

*Sample run (illustrative):*

```text
Category: billing
Reply: I'm sorry about the duplicate charge — I've flagged your account and we'll refund the extra payment right away.
```

<details><summary>Show solution</summary>

```python
def triage(message: str) -> str:
    prompt = (
        "Classify the message as billing, technical, or general, then write a "
        "one-sentence reply.\nFormat:\nCategory: <label>\nReply: <text>\n\n"
        f"Message: {message}"
    )
    return model.invoke(prompt).content

print(triage("I was charged twice for my subscription this month."))
```

This classify-then-respond pattern returns as a clean LCEL chain in later notebooks.

</details>

## Summary

- LangChain is a **framework for building applications around LLMs**, not a model itself.
- The ecosystem splits into `langchain-core` (abstractions), `langchain` (orchestration),
  provider packages like `langchain-google-genai`, `langchain-community`, and `langgraph`.
- `ChatGoogleGenerativeAI(model=..., temperature=...)` talks to Gemini;
  `.invoke()` returns an **`AIMessage`** whose text is in `.content`.
- Your key lives in `.env` and is loaded with `load_dotenv()`.

**Next:** [02 — Chat Models with Gemini](02_Chat_Models_with_Gemini.ipynb), where we go
deeper into messages, model parameters, streaming, and token usage.